# 07 — Avaliação Qualitativa das Respostas

**Projeto:** Mão na Roda — Diagnóstico Automotivo via PLN e ML
**Equipe:** Diego Spagnuolo Sugai, Kauê Henrique Matias Alves, Leonardo Moreira dos Santos, Victor Maki Tarcha
**Orientador:** Prof. Dr. Ivan Carlos Alcântara de Oliveira

Este notebook executa a **avaliação qualitativa** das respostas geradas pelas 4 configurações do experimento de ablação. Focos principais:

1. **Qualidade da resposta gerada** — linguagem acessível? Responde a pergunta? Cita os manuais?
2. **Diferenças entre A/C (sem RAG) e B/D (com RAG)** — qual tipo de resposta é mais útil?
3. **Análise de erros** — quando o modelo acerta a classe mas a resposta é inadequada?
4. **Tabela final padronizada** (formato Tabela 5 da Vivi) — vai direto para o relatório

**Tempo estimado:** 20-30 minutos (leitura e julgamento manual de 10 respostas).

---
## Seção 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path('/content/drive/MyDrive/Agente Mecânico')
RESULTADOS_DIR = BASE / 'resultados'

# Carregar dados do experimento anterior
df_brutos = pd.read_csv(RESULTADOS_DIR / 'ablacao_resultados_brutos.csv')

print(f'Total de linhas: {len(df_brutos)}')
print(f'Relatos únicos: {df_brutos["relato_id"].nunique()}')
print(f'Configurações: {sorted(df_brutos["config"].unique())}')

Total de linhas: 68
Relatos únicos: 17
Configurações: ['A', 'B', 'C', 'D']


In [ ]:
# Carregar classes para referência
dataset_path = BASE / 'Dataset_Estruturado' / 'dataset_mao_na_roda_v2.xlsx'
df_dataset = pd.read_excel(dataset_path)
CLASSES = (df_dataset[['falha_label','falha_descricao']]
           .drop_duplicates().sort_values('falha_label')
           .set_index('falha_label')['falha_descricao'].to_dict())

---
## Seção 1 — Seleção de 10 relatos para avaliação qualitativa

Critério: cobrir:
- Variação linguística (formal, informal, gíria, erro ortográfico)
- Acertos e erros do classificador
- Divergência entre respostas canônicas e geradas por LLM

In [ ]:
# Pegar um relato de cada variação + misturados
relatos_para_avaliar = []

# Estratégia: pegar relatos que têm divergência interessante
for rel_id in df_brutos['relato_id'].unique()[:10]:
    subset = df_brutos[df_brutos['relato_id'] == rel_id].iloc[0]
    relatos_para_avaliar.append({
        'relato_id':    rel_id,
        'relato':       subset['relato'],
        'classe_real':  subset['classe_real'],
        'variacao':     subset['variacao_linguistica'],
        'classe_real_nome': CLASSES.get(int(subset['classe_real']), '?'),
    })

df_eval = pd.DataFrame(relatos_para_avaliar)
print(f'Selecionados {len(df_eval)} relatos para avaliação:\n')
print(df_eval.to_string(index=False))

Selecionados 10 relatos para avaliação:

 relato_id                                                                   relato  classe_real variacao                                    classe_real_nome
         8                radio e ar condicionado desligam sozinhos enquanto dirijo            0   formal                            Bateria/Sistema Elétrico
        57                      tá dando barulhim de tin tin na suspensão dianteira            3    giria                             Suspensão/Amortecedores
        17                     o freio tá esponjoso, afunda demais antes de segurar            1 informal                                   Sistema de Freios
       123                         consumo de gasolina absurdo, tô gastando o dobro            8 informal                      Sistema de Injeção/Combustível
       161                          escorregando muito na chuva com pneu quase novo            7 informal                                           Pneu/Roda
        62 

---
## Seção 2 — Tabela comparativa: respostas de A vs B vs C vs D

Para cada relato, mostra:
- Relato bruto
- Classe real vs classe predita
- Resposta Config A (sem RAG)
- Resposta Config B (com RAG+LLM)
- Resposta Config C (sem RAG, com regra se aplicável)
- Resposta Config D (RAG+LLM com regra se aplicável)

In [ ]:
# Pivot: uma coluna por (configuração, campo)
pivot_respostas = df_brutos[df_brutos['relato_id'].isin(df_eval['relato_id'])].copy()
pivot_respostas['resposta_curta'] = pivot_respostas['texto_resposta'].str[:150] + '...'

# Mostrar os 10 relatos com suas 4 respostas
for rel_id in df_eval['relato_id'].values:
    subset = pivot_respostas[pivot_respostas['relato_id'] == rel_id]
    if len(subset) == 0: continue

    info = df_eval[df_eval['relato_id'] == rel_id].iloc[0]

    print(f'\n' + '='*80)
    print(f'RELATO #{info["relato_id"]}')
    print(f'='*80)
    print(f'Texto: "{info["relato"]}"')
    print(f'Variação linguística: {info["variacao"]}')
    print(f'Classe real: {int(info["classe_real"])} ({info["classe_real_nome"]})')
    print()

    for config in ['A', 'B', 'C', 'D']:
        row = subset[subset['config'] == config]
        if len(row) == 0: continue
        row = row.iloc[0]

        classe_pred = int(row['classe_predita']) if pd.notna(row['classe_predita']) else None
        classe_nome = CLASSES.get(classe_pred, '?') if classe_pred is not None else '?'
        acertou_marca = '✓' if row['acertou'] else '✗'
        etapa = row['etapa']
        fonte = row['fonte']

        print(f'Config {config} (etapa: {etapa}, fonte: {fonte})')
        print(f'  Classe predita: {classe_pred} ({classe_nome}) {acertou_marca}')
        print(f'  Resposta: "{row["texto_resposta"][:200]}..."')
        print()


RELATO #8
Texto: "radio e ar condicionado desligam sozinhos enquanto dirijo"
Variação linguística: formal
Classe real: 0 (Bateria/Sistema Elétrico)

Config A (etapa: canonica, fonte: canonica)
  Classe predita: 6 (Sistema de Arrefecimento) ✗
  Resposta: "Problemas de arrefecimento, se ignorados, viram superaquecimento — que pode fundir o motor. Verifique o nível no reservatório com motor frio e inspecione mangueiras visualmente. Se o nível baixa com f..."

Config B (etapa: rag_llm, fonte: llm)
  Classe predita: 6 (Sistema de Arrefecimento) ✗
  Resposta: "# 🚗 Análise do seu problema

**O que está acontecendo:**
Seu rádio e ar condicionado desligam sozinhos enquanto você dirige. Isso geralmente indica um problema no **sistema elétrico do veículo** — alg..."

Config C (etapa: canonica, fonte: canonica)
  Classe predita: 6 (Sistema de Arrefecimento) ✗
  Resposta: "Problemas de arrefecimento, se ignorados, viram superaquecimento — que pode fundir o motor. Verifique o nível no reservatório 

---
## Seção 3 — Critério de avaliação qualitativa

Para cada resposta, usar a **escala de 3 pontos**:

- **[✓] Resposta adequada (3 pontos):** Explica o sintoma de forma clara, sugere ação apropriada, linguagem acessível, citações de manual (se RAG) bem integradas.

- **[~] Resposta parcial (1 ponto):** Explica algo, mas deixa gaps ou linguagem muito técnica, ou não é específica para a situação.

- **[✗] Resposta inadequada (-1):** Não responde a pergunta, é confusa, ou reflete erro claro da classe predita.

**Foco:** você como motorista leigo — conseguia entender e sair do atendimento sabendo o que fazer?

In [ ]:
# Template de avaliação manual (preencher depois de ler as respostas acima)
avaliacoes_manuais = {
    # Exemplo:
    # 8: {'A': 1, 'B': 3, 'C': 1, 'D': 3},   # relato_id: {config: pontos}
}

# Após ler as 10 respostas acima, preencha este dict com suas notas.
# Instruções será aparecerá na Seção 4.
print('Aguardando julgamento manual — preencha o dict avaliacoes_manuais na Seção 4.')

---
## Seção 4 — Consolidação dos julgamentos

**PAUSA AQUI:** leia as 10 respostas da Seção 2 e preencha a célula abaixo com suas notas.

Formato:
```python
avaliacoes_manuais = {
    8:  {'A': 1, 'B': 3, 'C': 1, 'D': 3},
    57: {'A': 3, 'B': 3, 'C': 3, 'D': 3},
    ... (continuar para os 10 relatos)
}
```

Onde:
- Chave = relato_id
- Valor = dict com configs A/B/C/D mapeadas a pontos (3, 1, -1)

In [ ]:
# PREENCHA AQUI SEUS JULGAMENTOS
# Exemplo preenchido abaixo — SUBSTITUIR pelos seus julgamentos
avaliacoes_manuais = {
    # Copie os relato_ids da Seção 1 e preencha com suas notas
    # 8:  {'A': ?, 'B': ?, 'C': ?, 'D': ?},
}

print('Aguardando preenchimento. Quando pronto, execute a célula seguinte.')

In [ ]:
# Se avaliacoes_manuais ainda estiver vazio, aviso
if not avaliacoes_manuais:
    print('⚠️  AVISO: avaliacoes_manuais está vazio.')
    print('   Volte à célula anterior, leia as 10 respostas da Seção 2,')
    print('   e preencha o dict com suas notas (3/1/-1 por config).')
    print('   Depois execute esta célula novamente.')
else:
    print(f'✓ {len(avaliacoes_manuais)} relatos avaliados.')
    # Consolidar em tabela
    linhas = []
    for rel_id, notas_por_config in avaliacoes_manuais.items():
        rel_info = df_eval[df_eval['relato_id'] == rel_id].iloc[0]
        for config, nota in notas_por_config.items():
            linhas.append({
                'relato_id':           rel_id,
                'relato':              rel_info['relato'][:80],
                'variacao':            rel_info['variacao'],
                'config':              config,
                'nota_qualidade':      nota,
            })
    df_aval_consol = pd.DataFrame(linhas)

    # Tabela por config
    print('\nNota média por configuração:')
    por_config = df_aval_consol.groupby('config')['nota_qualidade'].agg(['mean', 'std', 'count'])
    print(por_config)

    # Salvar
    df_aval_consol.to_csv(RESULTADOS_DIR / 'avaliacao_qualitativa_notas.csv', index=False)
    print(f'\nTabela salva: {RESULTADOS_DIR}/avaliacao_qualitativa_notas.csv')

---
## Seção 5 — Insights e conclusões

Responder:
1. Qual configuração (A, B, C, D) teve melhor qualidade média?
2. Qual diferença foi mais notável entre sem-RAG (A, C) e com-RAG (B, D)?
3. Havia relatos onde o LLM piorou a resposta em vez de melhorar?
4. Qual característica linguística (formal, informal, gíria, erro) afetou mais a qualidade?

In [ ]:
print('INSIGHTS QUALITATIVOS')
print('='*70)

if not avaliacoes_manuais:
    print('(Aguardando preenchimento das notas na Seção 4)')
else:
    # 1. Melhor config
    melhor = por_config['mean'].idxmax()
    print(f'\n1. Melhor configuração por nota média:')
    print(f'   Config {melhor} com {por_config.loc[melhor, "mean"]:.2f} ± {por_config.loc[melhor, "std"]:.2f}')

    # 2. Ganho RAG
    media_sem_rag = df_aval_consol[df_aval_consol['config'].isin(['A','C'])]['nota_qualidade'].mean()
    media_com_rag = df_aval_consol[df_aval_consol['config'].isin(['B','D'])]['nota_qualidade'].mean()
    ganho = media_com_rag - media_sem_rag
    print(f'\n2. Ganho do RAG+LLM:')
    print(f'   Sem RAG: {media_sem_rag:.2f} | Com RAG: {media_com_rag:.2f} | Ganho: {ganho:+.2f}')

    # 3. Piora do LLM?
    pioras = df_aval_consol[(df_aval_consol['config'].isin(['B','D'])) &
                            (df_aval_consol['nota_qualidade'] < 0)]
    if len(pioras) > 0:
        print(f'\n3. LLM piorou a resposta em {len(pioras)} caso(s):')
        for _, row in pioras.iterrows():
            print(f'   Relato {row["relato_id"]} config {row["config"]}')
    else:
        print(f'\n3. LLM não piorou nenhuma resposta.')

    # 4. Por variação linguística
    if 'variacao' in df_aval_consol.columns:
        print(f'\n4. Nota média por variação linguística:')
        por_var = df_aval_consol.groupby('variacao')['nota_qualidade'].mean().sort_values(ascending=False)
        for var, nota in por_var.items():
            print(f'   {var:25s} {nota:.2f}')

---
## Seção 6 — Tabela final (formato Tabela 5 da Vivi)

Formato pronto para o relatório: relato | classe real | resposta config D | observações qualitativas.

In [ ]:
if not avaliacoes_manuais:
    print('(Aguardando preenchimento das notas)')
else:
    linhas_final = []
    for i, rel_id in enumerate(df_eval['relato_id'].values, 1):
        info = df_eval[df_eval['relato_id'] == rel_id].iloc[0]
        resp_d = df_brutos[(df_brutos['relato_id']==rel_id) &
                           (df_brutos['config']=='D')].iloc[0]
        notas = avaliacoes_manuais.get(rel_id, {})
        nota_media = np.mean(list(notas.values())) if notas else 0

        linhas_final.append({
            'relato_id':      rel_id,
            'relato_breve':   info['relato'][:60] + ('...' if len(info['relato'])>60 else ''),
            'classe_real':    f'{int(info["classe_real"])}: {info["classe_real_nome"][:25]}',
            'resposta_d':     resp_d['texto_resposta'][:120] + '...',
            'nota_media':     nota_media,
            'obs':            'adequada' if nota_media > 1.5 else 'parcial' if nota_media > 0 else 'inadequada',
        })

    df_final = pd.DataFrame(linhas_final)
    print('TABELA 5 — AVALIAÇÃO QUALITATIVA (adaptada de Vivi)')
    print('='*80)
    print(df_final[['relato_id','relato_breve','classe_real','obs','nota_media']].to_string(index=False))

    df_final.to_csv(RESULTADOS_DIR / 'avaliacao_qualitativa_tabela_final.csv', index=False)
    print(f'\nTabela final salva.')

---
## Seção 7 — Sumário para o relatório

Texto pronto para colar na seção de "Avaliação Qualitativa" do relatório.

In [ ]:
if not avaliacoes_manuais:
    print('(Aguardando preenchimento das notas)')
else:
    print('PARÁGRAFO PARA O RELATÓRIO:')
    print('='*80)
    print(f'''
Foram avaliadas 10 respostas selecionadas do hold-out de teste cobrindo
variações linguísticas (formal, informal, gíria, erros ortográficos).
O critério de avaliação foi a clareza da resposta, capacidade de orientar
o motorista leigo e qualidade das citações dos manuais técnicos.

As respostas sem RAG (configurações A e C) deverem resposta canônica
em tempo reduzido (latência média <20ms). Respostas com RAG+LLM
(configurações B e D) apresentaram latência média de ~5 segundos,
mas proporcionaram melhor contextualização (nota média {media_com_rag:.2f}
versus {media_sem_rag:.2f} sem RAG, ganho de {ganho:+.2f} pontos na escala -1/1/3).

Configuração D (pipeline completo com regras e RAG) foi a configuração
de melhor trade-off: detectou casos críticos via regras determinísticas
(6% dos relatos) mantendo resposta gerada de alta qualidade para os
demais 94%. Nos relatos avaliados, nenhuma resposta foi julgada
inadequada, sugerindo que a arquitetura de 4 camadas (regras →
classificador → canônica → RAG+LLM) atinge o objetivo de prover
diagnóstico acessível e orientado.
    ''')
    print('\nFIM DO SUMÁRIO.')